In [14]:
import os
import pickle
from pathlib import Path
import pandas as pd

FILE_PATH = os.path.dirname(os.path.abspath("."))
os.chdir(FILE_PATH)
from src.bm25 import BM25Search
from src.semantic import SemanticSearch
from src.download_data import download_data
os.chdir(f"{FILE_PATH}/notebooks")

download_data()

CATEGORY = "Appliances"
PROCESSED_DATA_DIR = Path("../data/processed")
with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_product_documents.pkl", "rb") as f:
    documents = pickle.load(f)

with open(f"{PROCESSED_DATA_DIR}/{CATEGORY}_doc_ids.pkl", "rb") as f:
    doc_ids = pickle.load(f)

Review data for Appliances already downloaded
Meta data for Appliances already downloaded
Merged data for Appliances is ready
Document id for Appliances is ready
Document id for Appliances is ready


In [27]:
import duckdb

PROCESSED_DATA_DIR = Path("../data/processed")
product_data_file = "Appliances_products.parquet"

c2 = duckdb.connect()
products = c2.execute(f"SELECT * FROM read_parquet('{PROCESSED_DATA_DIR}/{product_data_file}')").df()

In [28]:
print(products.keys())

Index(['parent_asin', 'product_title', 'main_category', 'store', 'price',
       'avg_rating', 'reviews', 'review_titles', 'helpful_votes'],
      dtype='str')


#### It took me approximately 25 minutes to build index from scratch (mostly for SemanticSearch) on M1 MacBook Air with 16GB RAM

In [17]:
bm25_engine = BM25Search(documents)
semantic_engine = SemanticSearch(documents)

load bm25 index
done
load tokenized products
done


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1768.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


load faiss index
done


#### I asked ChatGPT to "generate 21 queries for a amazon product recommendation system for Appliances (7 easy keyword-based queries, 7 medium semantic based queries, 7 complex queries)"
- stored in `data/queries.csv`

In [5]:
queries = pd.read_csv("../data/queries.csv")
queries

,difficulty,text
0,easy,coffee maker with grinder
1,easy,stainless steel toaster 2 slice
2,easy,air fryer 5 quart
3,easy,electric kettle glass
4,easy,portable mini fridge
5,easy,vacuum cleaner cordless
6,easy,microwave oven compact
7,medium,appliance to quickly make coffee in the morning
8,medium,something to reheat food fast in a small kitchen
9,medium,device to keep drinks cold while working at desk


In [29]:
# Run queries
for d, q in zip(queries['difficulty'], queries['text']):
    print("=" * 80)
    print(f"DIFFICULTY: {d}\nQUERY: {q}\n")

    print("BM25 top results:")
    for i, (index, score) in enumerate(bm25_engine.search(q)):
        product_asin = doc_ids[index]
        print(f"{i+1}. ({score:.3f}) {products.loc[products['parent_asin'] == product_asin]['product_title']}")

    print("\nEmbedding search top results:")
    for i, (index, score) in enumerate(semantic_engine.search(q)):
        product_asin = doc_ids[index]
        print(f"{i+1}. ({score:.3f}) {products.loc[products['parent_asin'] == product_asin]['product_title']}")

    print()

DIFFICULTY: easy
QUERY: coffee maker with grinder

BM25 top results:
1. (21.263) 5673    Coffee Gift Sets - Includes French Press Coffe...
Name: product_title, dtype: str
2. (20.369) 39624    10-Cup Drip Coffee Maker, Grind and Brew Autom...
Name: product_title, dtype: str
3. (16.853) 56939    Ninja 12-Tablespoon Coffee & Spice Grinder – N...
Name: product_title, dtype: str
4. (16.809) 51737    Manual Hand Coffee Grinder Mini Transparent Wi...
Name: product_title, dtype: str
5. (16.463) 4906    Pour Over Coffee Dripper, Stainless Steel Pour...
Name: product_title, dtype: str

Embedding search top results:
1. (0.641) 39624    10-Cup Drip Coffee Maker, Grind and Brew Autom...
Name: product_title, dtype: str
2. (0.575) 51737    Manual Hand Coffee Grinder Mini Transparent Wi...
Name: product_title, dtype: str
3. (0.569) 77412    Delonghi 7313216831 Complete Grinder
Name: product_title, dtype: str
4. (0.531) 86780    Coffeeduck for Philips Senseo New Generation C...
Name: product_title, dty

In [41]:
qq = ["Cheapest oven"]
for q in qq:
    print("=" * 80)
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    for i, (index, score) in enumerate(bm25_engine.search(q)):
        product_asin = doc_ids[index]
        print(f"{i+1}. ({score:.3f}) {products.loc[products['parent_asin'] == product_asin]['product_title']}")

    print("\nEmbedding search top results:")
    for i, (index, score) in enumerate(semantic_engine.search(q)):
        product_asin = doc_ids[index]
        print(f"{i+1}. ({score:.3f}) {products.loc[products['parent_asin'] == product_asin]['product_title']}")

    print()



QUERY: Cheapest oven

BM25 top results:
1. (13.526) 32556    Samsung DG97-00083A Oven Lamp Bulb Assembly
Name: product_title, dtype: str
2. (11.880) 83056    Electric Oven Knob Kit by Ez-Flo International...
Name: product_title, dtype: str
3. (11.829) 27710    9758079 Oven Spark Igniter Replacement for Whi...
Name: product_title, dtype: str
4. (11.106) 30848    Whirlpool 8273004 Burner
Name: product_title, dtype: str
5. (10.829) 15334    GE WB27T11349 Control Oven To9 Elec
Name: product_title, dtype: str

Embedding search top results:
1. (0.632) 20124    Miele Classic Design H4884BP 30 Single Electri...
Name: product_title, dtype: str
2. (0.618) 30    COSMO COS-30ESWC 30 in. 5 cu. ft. Single Elect...
Name: product_title, dtype: str
3. (0.615) 77067    Supplying Demand W10308477 9758519 Electric Ra...
Name: product_title, dtype: str
4. (0.612) 70560    Range Oven Bake Lower Unit Heating Element CH7...
Name: product_title, dtype: str
5. (0.609) 50812    GE PB975BMBB ProfileTM 30" Free-St